In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix 
from sklearn.metrics import precision_recall_fscore_support
import torch
from datasets import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    Trainer,
    TrainingArguments
)

In [14]:
df = pd.read_csv("csvs/small_translation_dataset.csv")
X = df["text"].values
y = df["label"].values

In [15]:
dataset = Dataset.from_dict({
    "text": X.tolist(),
    "label": y.tolist()
})
dataset = dataset.train_test_split(test_size=0.2)
train_dataset = dataset["train"]
test_dataset = dataset["test"]

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )
    
def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512)


print(dataset)
# TOKENIZED_DATA_FILE = "data/sp-en/sp-machine-translation-10k_tokenized.parquet"
# OUTPUT_DIR = "./data/sp-en/sp-machine-translation-10k-out"
    
# tokenized_ds = (dataset.map(tokenize_batch, batched=True).rename_column("label", "labels"))
# tokenized_ds.to_parquet(TOKENIZED_DATA_FILE)


# train_dataset = train_dataset.map(tokenize, batched=True)
# test_dataset = test_dataset.map(tokenize, batched=True)


# tokenized_ds = tokenized_ds.cast_column(
#     "labels", 
#     ClassLabel(num_classes=2, names=["human", "machine"])
# )

# tokenized_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# split = tokenized_ds.train_test_split(
#     test_size=0.1, 
#     seed=1865, 
#     stratify_by_column="labels"
# )

# train_ds = split["train"]
# test_ds = split["test"]

# # check distribution
# print("Train set distribution:", torch.bincount(torch.tensor(train_ds["labels"])))
# print("Test set distribution:", torch.bincount(torch.tensor(test_ds["labels"])))

# data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
# train_ds_full = train_ds



model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01
)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 640
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 160
    })
})


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 5558.81it/s]
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
# train_dataset = dataset["train"]
# test_dataset = dataset["test"]


# -----------------------
# 2. Tokenization
# -----------------------
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

# Set format for PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# -----------------------
# 3. Load model
# -----------------------
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
)

# OPTIONAL (recommended for small data): freeze base model
# for param in model.roberta.parameters():
#     param.requires_grad = False

# -----------------------
# 4. Metrics
# -----------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

# -----------------------
# 5. Training arguments
# -----------------------
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.00,

    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

# -----------------------
# 6. Trainer
# -----------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    # tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# -----------------------
# 7. Train
# -----------------------
trainer.train()

# -----------------------
# 8. Evaluate
# -----------------------
# results = trainer.evaluate()
# print()
# print()
# print("Evaluation results:")
# print(results)

# # -----------------------
# # 9. Predict function
# # -----------------------
# def predict(text):
#     inputs = tokenizer(
#         text,
#         return_tensors="pt",
#         truncation=True,
#         padding=True,
#         max_length=256
#     )

#     with torch.no_grad():
#         outputs = model(**inputs)

#     probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
#     pred = torch.argmax(probs, dim=1).item()

#     return {
#         "label": pred,
#         "confidence": probs[0][pred].item()
#     }

# # Example usage:
# print(predict("This is a test sentence."))

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 6639.49it/s]
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/opt/miniconda3/envs/machineLearning/lib/py

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.702303,0.694009,0.475000,0.644068,0.475000,1.000000
2,0.696660,0.692107,0.525000,0.000000,0.000000,0.000000
3,0.702945,0.694927,0.475000,0.644068,0.475000,1.000000
4,0.697359,0.694619,0.475000,0.644068,0.475000,1.000000
5,0.694594,0.695029,0.475000,0.644068,0.475000,1.000000


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.22s/it]
/opt/miniconda3/envs/machineLearning/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/opt/miniconda3/envs/machineLearning/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.38s/it]
/opt/miniconda3/envs/machineLearning/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|████████

TrainOutput(global_step=400, training_loss=0.6987721443176269, metrics={'train_runtime': 139.5163, 'train_samples_per_second': 22.936, 'train_steps_per_second': 2.867, 'total_flos': 420977688576000.0, 'train_loss': 0.6987721443176269, 'epoch': 5.0})

In [19]:
print(model.config)

RobertaConfig {
  "add_cross_attention": false,
  "architectures": [
    "RobertaForSequenceClassification"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 1,
  "problem_type": "single_label_classification",
  "tie_word_embeddings": true,
  "transformers_version": "5.3.0",
  "type_vocab_size": 1,
  "use_cache": false,
  "vocab_size": 50265
}



In [18]:
trainer.evaluate()

/opt/miniconda3/envs/machineLearning/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


RuntimeError: on_train_begin must be called before on_evaluate

In [15]:
import transformers
print(transformers.__version__)
print(transformers.__file__)

5.3.0
/opt/miniconda3/envs/machineLearning/lib/python3.10/site-packages/transformers/__init__.py


In [ ]:
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    return {"accuracy": accuracy_score(labels, preds)}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()
trainer.evaluate()

In [ ]:
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)
    return probs.detach().numpy()